## Structure of this notebook

- **(1) Preparations**
  - **(1.1)** Import of all necessary packages
  - **(1.2)** Loading raw data of ZWN reports

- **(2) Clean raw data of ZWN reports**

- **(3) Convert the DataFrame to GeoDataFrame for spatial analysis**

- **(4) Save as .gpkg to data-folder**

In [1]:
# (1) Preparations

# (1.1) Import of all necessary packages
import pandas as pd
import geopandas as gpd
from shapely import wkt

# (1.2) Loading raw data of ZWN reports
zh_df = pd.read_csv("../data/raw/stzh.zwn_meldungen_p.csv")

print("Data loaded successfull")

Data loaded successfull


In [2]:
# (2) Clean raw data of ZWN reports
# Deleting unnecessary columns
zh_df = zh_df.drop(columns=["objectid", "agency_sent_datetime", "service_code", "media_url"],
    errors="ignore"
) 

# Parse date-time columns
zh_df["requested_datetime"] = pd.to_datetime(
    zh_df["requested_datetime"],
    format="%Y-%m-%dT%H:%M:%S"
)

zh_df["updated_datetime"] = pd.to_datetime(
    zh_df["updated_datetime"],
    format="%Y-%m-%dT%H:%M:%S"
)

# Check for NaN values
print("NaN values found:")
display(zh_df.isna().sum().sort_values(ascending=False))

# Fill missing values in text columns
text_columns = ["title", "detail", "service_notice"]

zh_df[text_columns] = zh_df[text_columns].fillna("")

NaN values found:


service_notice        862
title                   2
detail                  2
e                       0
service_request_id      0
requested_datetime      0
updated_datetime        0
status                  0
service_name            0
n                       0
userid                  0
interface_used          0
description             0
url                     0
geometry                0
dtype: int64

In [3]:
# (3) Convert the DataFrame to GeoDataFrame for spatial analysis
zh_df["geometry"] = zh_df["geometry"].apply(wkt.loads)

zh_gdf = gpd.GeoDataFrame(
    zh_df, 
    geometry="geometry",
    crs="EPSG:2056"
)

In [4]:
# (4) Save as .gpkg to data-folder
output_gpkg = "../data/cleaned/stzh.zwn_meldungen_p_cleaned.gpkg"

# Save as GeoPackage
zh_gdf.to_file(
    output_gpkg,
    layer="meldungen",
    driver="GPKG"
)